In [91]:
    #Input monodromy decomposition. Should be an array of even length with only positive integers.
class Punctured_Torus_Bundle():
        
        def __init__(self, monodromy):
            #Initialize all variables used in class
            self.twist = monodromy 
            self.period_array = self.make_period_array()
            self.period = len(self.period_array)
            self.matrices = self.make_matrices()
            self.homology_relators = self.make_homology_relators()
            self.reduced_homology_relators = self.make_reduced_homology_relators()
            self.obstruction = self.make_obstruction_classes()
            self.R = self.make_unreduced_generators()
            self.S = self.make_reduced_generators()
            self.C = self.make_reduced_cross_generators()
            self.unred_gluings_dicts = self.make_unred_gluing_dicts()
            self.hom_dicts = self.make_hom_dicts()
            self.cross_hom_dicts = self.make_cross_hom_dicts()
            self.ideals = self.make_ideals()
            self.cross_ideals = self.make_cross_ideals()
            
            
            

        #Gluing type for each tetrahedron in the triangulaiton. 
        def make_period_array(self):
            
            period_array = []
            
            for twist in range(len(self.twist)):
                for i in range(self.twist[twist]):
                    period_array.append(twist % 2)
                    
            return period_array


        #Compute psl-obstruction classes
        #According to Lefshetz duality, the relative cohomology H2 (M, boundary(M)) is isomorphic to the homology H1(M). 
        #The fundamental group of M is the semidirect product Z X F2. Where tx=phi(x)t, ty=phi(y)t. Abelianizing gives <t,x,y|x=phi(x),y=phi(y)>.
        #We will therefore first compute phi(x) and phi(y).
        def make_matrices(self):
            
            R = Matrix([[1,1],[0,1]])
            L = Matrix([[1,0],[1,1]])

            #We construct all intermediate matrices in the triangulation which will be useful later.
            M=[matrix.identity(2)]
            for i in self.period_array:
                if i == 0:
                    M.append(M[-1] * R)
                else:
                    M.append(M[-1] * L)

            return M
        
        
        #Here we determine the relators for the homology.
        def make_homology_relators(self):
            
            N = (self.matrices[-1]-matrix.identity(2)).elementary_divisors()
            
            return N
        
        #We only care about the homology relators up to mod 2. Therefore, if a generator of the homology is cyclic with odd order, we do not care about it.
        def make_reduced_homology_relators(self):
            return [(self.homology_relators[0] + 1) % 2, (self.homology_relators[1] + 1) % 2]
        

        #We now construct the obstruction classes. We only include the obstruction classes arising from the surface-group if they are non-trivial.
        def make_obstruction_classes(self):
            obs_triv = []
            for i in range(self.period):
                obs_triv.append([1,1,1,1,1,1])


            #We construct the obstruction class corresponding to the meridian of the periferal subgroup.
            obs_fund = [[1,-1,1,1,1,1]]
            for i in range(self.period - 1):
                if self.period_array[i] == 0:
                    obs_fund.append([-1,1,1,1,1,1])
                else:
                    obs_fund.append([1,-1,1,1,1,1])


            #We construct the obstruction class corresponding to each surface-group generator:
            obs_x = [[1,1,-1,1,1,1],[1,1,-1,1,1,1]]
            
            for i in range(self.period-2):
                obs_x.append([1,1,1,1,1,1])

            obs_y = [[1,1,-1,1,1,1],[1,1,-1,1,1,1]]
            for i in range(self.period-2):
                obs_y.insert(i+1, [1,1,1,1,1,1])
                
            
            #Here we construct all the possible product obstruction classes.
            obs_mx = []
            for period in range(self.period):
                obs_temp = []
                for i, j in zip(obs_fund[period], obs_x[period]):
                    obs_temp.append((i*j))
                        
                obs_mx.append(obs_temp)
                
            obs_my = []
            for period in range(self.period):
                obs_temp = []
                for i, j in zip(obs_fund[period], obs_y[period]):
                    obs_temp.append((i*j))
                        
                obs_my.append(obs_temp)
            
            obs_xy = []
            for period in range(self.period):
                obs_temp = []
                for i, j in zip(obs_x[period], obs_y[period]):
                    obs_temp.append((i*j))
                        
                obs_xy.append(obs_temp)
                
            obs_mxy = []
            for period in range(self.period):
                obs_temp = []
                for i, j, k in zip(obs_fund[period],obs_x[period], obs_y[period]):
                    obs_temp.append((i*j*k))
                        
                obs_mxy.append(obs_temp)
            


            obstruction = [obs_triv, obs_fund]
            if (self.reduced_homology_relators[0] != 0):
                obstruction.append(obs_x)
                obstruction.append(obs_mx)
        
            if (self.reduced_homology_relators[1] != 0):
                obstruction.append(obs_y)
                obstruction.append(obs_my)
                                   
            if (self.reduced_homology_relators[0] != 0 and self.reduced_homology_relators[1] != 0):
                obstruction.append(obs_xy)
                obstruction.append(obs_mxy)
                

            return obstruction

        #We are now ready to construct the Ptolemy equations


        #First prepare unreduced edge-equations for the Ptolemy Variety
        def make_unreduced_generators(self):
            gens = 'y01_0, y02_0, y03_0, y12_0, y13_0, y23_0'
            for i in range(self.period - 1):
                gens = gens + ' ,'+'y01_{index}, y02_{index}, y03_{index}, y12_{index}, y13_{index}, y23_{index}'.format( index = i + 1 )

            R=PolynomialRing(QQbar, 6*self.period, names = gens)
#             R.inject_variables()
            
            return R

        #Euler Characteristic = 0 implies number of edges = number of tetrahedra
        #Prepare edge-variables for reduced Ptolemy Variety
        def make_reduced_generators(self):
            gens = 'x0'
            for i in range(self.period - 1):
                gens = gens + ' ,' + f'x{i + 1}'

            gens = gens + ' ,'+'const'
            
            S=PolynomialRing(QQbar, self.period + 1, names = gens)
            S.inject_variables()
            
            return S
        
        def make_reduced_cross_generators(self):
            gens = 'c0, z0, zp0, zpp0'
            for i in range(self.period - 1):
                gens = gens + ' ,' + f'c{i + 1}'
                gens = gens + ' ,' + f'z{i + 1}, zp{i + 1}, zpp{i + 1}'

            gens = gens + ' ,'+'c'
            
            C=PolynomialRing(QQbar, 4 * self.period + 1, names = gens)
            C.inject_variables()
            
            return C


        # Establish the gluing set so that we can map the unreduced ideal to the reduced ideal.
        def make_unred_gluing_dicts(self):
            unred_gluings_dicts = []


            for obs in self.obstruction:
                glue_dict = {}
                fglue_dict = {}

                for k in range(self.period):
                    
                    if (self.period_array[k]) == 0:  
                        glue_dict[self.R(f'y03_{k}')] = [self.R(f' {obs[k][2]*obs[(k+1) % self.period][1]} * y02_{(k+1) % self.period}'), 
                                                         self.R(f' {obs[k][2]*obs[(k+1) % self.period][4]} * y13_{(k+1) % self.period}')]
                        glue_dict[self.R(f'y01_{k}')] = [self.R(f' {obs[k][0]*obs[(k+1) % self.period][0]} * y01_{(k+1) % self.period}')]
                        glue_dict[self.R(f'y23_{k}')] = [self.R(f' {obs[k][5]*obs[(k+1) % self.period][5]} * y23_{(k+1) % self.period}')]
                        glue_dict[self.R(f'y02_{k}')] = [self.R(f' {obs[k][1]*obs[(k+1) % self.period][3]} * y12_{(k+1) % self.period}')]
                        glue_dict[self.R(f'y13_{k}')] = [self.R(f' {obs[k][4]*obs[(k+1) % self.period][3]} * y12_{(k+1) % self.period}')]

                    else:
                        glue_dict[self.R(f'y03_{k}')] = [self.R(f' {obs[k][2]*obs[(k+1) % self.period][0]} * y01_{(k+1) % self.period}'), 
                                                         self.R(f' {obs[k][2]*obs[(k+1) % self.period][5]} * y23_{(k+1) % self.period}')]
                        glue_dict[self.R(f'y13_{k}')] = [self.R(f' {obs[k][4]*obs[(k+1) % self.period][4]} * y13_{(k+1) % self.period}')]
                        glue_dict[self.R(f'y02_{k}')] = [self.R(f' {obs[k][1]*obs[(k+1) % self.period][1]} * y02_{(k+1) % self.period}')]
                        glue_dict[self.R(f'y01_{k}')] = [self.R(f'-{obs[k][0]*obs[(k+1) % self.period][3]} * y12_{(k+1) % self.period}')]
                        glue_dict[self.R(f'y23_{k}')] = [self.R(f'-{obs[k][5]*obs[(k+1) % self.period][3]} * y12_{(k+1) % self.period}')]


                #The dictionary does not recognize negative values. We make a new dictionary which maps negative edges to the respective negative values.
                for key in glue_dict:
                    min_out = []
                    for var in glue_dict[key]:
                        min_out.append(-var)
                        

                    fglue_dict[key] = glue_dict[key]
                    fglue_dict[-key] = min_out

                unred_gluings_dicts.append(fglue_dict)
            unred_gluings_dicts[0][self.R.gens()[2]]
            
            return unred_gluings_dicts


        #Now we generate a dictionary which maps the unreduced edge variables to the reduced edge variables
        
        def make_hom_dicts(self):

            #The reduction will always end at the diagonal y12_i.
            ends = []
            for i in range(self.period):
                ends.append(self.R.gens()[3 + 6 * i])
                ends.append(self.R.gens()[3 + 6 * i])


            hom_dicts = []
            for obs in range(len(self.obstruction)):
                hom_dict = {}

                for i in range(self.period):

                    #All gluings will begin at the diagonals y03_i. In the generator set, this always has an index 2 + 6i
                    edge_class = 2 + 6 * i
                    hom_dict[self.R.gens()[edge_class]] = self.S(f'x{i}')

                    #We itterate through the gluing dictionary so that all edges which are glued together map to the same value xi.
                    #Extra care is taken since the dictionary does not recognize negative values.

                    end = self.unred_gluings_dicts[obs][self.R.gens()[edge_class]][0]
                    while end not in ends and -end not in ends:
                        if end in self.R.gens():
                            hom_dict[end] = hom_dict[self.R.gens()[edge_class]]
                        else:
                            hom_dict[-end] = -hom_dict[self.R.gens()[edge_class]]

                        end = self.unred_gluings_dicts[obs][end][0]

                    if end in self.R.gens():
                        hom_dict[end] = hom_dict[self.R.gens()[edge_class]]
                    else:
                        hom_dict[-end] = -hom_dict[self.R.gens()[edge_class]]

                    end = self.unred_gluings_dicts[obs][self.R.gens()[edge_class]][1]

                    while end not in ends and -end not in ends:
                        if end in self.R.gens():
                            hom_dict[end] = hom_dict[self.R.gens()[edge_class]]
                        else:
                            hom_dict[-end] = -hom_dict[self.R.gens()[edge_class]]
                        end = self.unred_gluings_dicts[obs][end][0]


                    if end in self.R.gens():
                        hom_dict[end] = hom_dict[self.R.gens()[edge_class]]
                    else:
                        hom_dict[-end] = -hom_dict[self.R.gens()[edge_class]]            


                hom_dicts.append(hom_dict)

            return hom_dicts
        
        def make_cross_hom_dicts(self):

            #The reduction will always end at the diagonal y12_i.
            ends = []
            for i in range(self.period):
                ends.append(self.R.gens()[3 + 6 * i])
                ends.append(self.R.gens()[3 + 6 * i])


            hom_dicts = []
            for obs in range(len(self.obstruction)):
                hom_dict = {}

                for i in range(self.period):

                    #All gluings will begin at the diagonals y03_i. In the generator set, this always has an index 2 + 6i
                    edge_class = 2 + 6 * i
                    hom_dict[self.R.gens()[edge_class]] = self.C(f'c{i}')

                    #We itterate through the gluing dictionary so that all edges which are glued together map to the same value xi.
                    #Extra care is taken since the dictionary does not recognize negative values.

                    end = self.unred_gluings_dicts[obs][self.R.gens()[edge_class]][0]
                    while end not in ends and -end not in ends:
                        if end in self.R.gens():
                            hom_dict[end] = hom_dict[self.R.gens()[edge_class]]
                        else:
                            hom_dict[-end] = -hom_dict[self.R.gens()[edge_class]]

                        end = self.unred_gluings_dicts[obs][end][0]

                    if end in self.R.gens():
                        hom_dict[end] = hom_dict[self.R.gens()[edge_class]]
                    else:
                        hom_dict[-end] = -hom_dict[self.R.gens()[edge_class]]

                    end = self.unred_gluings_dicts[obs][self.R.gens()[edge_class]][1]

                    while end not in ends and -end not in ends:
                        if end in self.R.gens():
                            hom_dict[end] = hom_dict[self.R.gens()[edge_class]]
                        else:
                            hom_dict[-end] = -hom_dict[self.R.gens()[edge_class]]
                        end = self.unred_gluings_dicts[obs][end][0]


                    if end in self.R.gens():
                        hom_dict[end] = hom_dict[self.R.gens()[edge_class]]
                    else:
                        hom_dict[-end] = -hom_dict[self.R.gens()[edge_class]]            


                hom_dicts.append(hom_dict)

            return hom_dicts
        
        
        def change_variables(self, obstruction = 0, var = ''):
            if var in self.R.gens():
                return self.hom_dicts[obstruction][var]
            elif self.R(var) in self.R.gens():
                return self.hom_dicts[obstruction][self.R(var)]
            else:
                raise ValueError (f'{var} is not a recognized variable')
                
                
        def change_cross_variables(self, obstruction = 0, var = ''):
            if var in self.R.gens():
                return self.cross_hom_dicts[obstruction][var]
            elif self.R(var) in self.R.gens():
                return self.cross_hom_dicts[obstruction][self.R(var)]
            else:
                raise ValueError (f'{var} is not a recognized variable')
            


        #Now we map the gluing equations to our reduced dimensional space and generate the variety over which our solutions exist.
        
        def make_ideals(self):
            ideals = []
            non_zero_polynomial = const
            
            for gen in range(self.period):
                non_zero_polynomial = non_zero_polynomial * self.S(f'x{gen}')
                
            for obs in range(len(self.obstruction)):
                ptolemy = []
                
                for i in range (self.period):
                    ptolemy.append(  self.change_variables(obs, f'y03_{i}') * self.change_variables(obs, f'y12_{i}') 
                                   + self.change_variables(obs, f'y01_{i}') * self.change_variables(obs, f'y23_{i}')
                                   - self.change_variables(obs, f'y02_{i}') * self.change_variables(obs, f'y13_{i}'))
                
                ptolemy.append(non_zero_polynomial - 1)
                
                
                ptolemy.append(x0 - 1)

                ideals.append(self.S.ideal(ptolemy))
                
            return ideals
        
        
        def make_cross_ideals(self):
            ideals = []
            non_zero_polynomial = c
            
            for gen in range(self.period):
                non_zero_polynomial = non_zero_polynomial * self.C(f'c{gen}')
                
            for obs in range(len(self.obstruction)):
                ptolemy = []
                
                for i in range (self.period):
                    ptolemy.append(  self.change_cross_variables(obs, f'y03_{i}') * self.change_cross_variables(obs, f'y12_{i}') 
                                   + self.change_cross_variables(obs, f'y01_{i}') * self.change_cross_variables(obs, f'y23_{i}')
                                   - self.change_cross_variables(obs, f'y02_{i}') * self.change_cross_variables(obs, f'y13_{i}'))
                    
                    ptolemy.append( self.obstruction[obs][i][0] * self.C(f'z{i}')   * self.change_cross_variables(obs, f'y03_{i}') * self.change_cross_variables(obs, f'y12_{i}') - self.change_cross_variables(obs, f'y02_{i}') * self.change_cross_variables(obs, f'y13_{i}'))
                    ptolemy.append(-self.obstruction[obs][i][1] * self.C(f'zp{i}')  * self.change_cross_variables(obs, f'y01_{i}') * self.change_cross_variables(obs, f'y23_{i}') - self.change_cross_variables(obs, f'y03_{i}') * self.change_cross_variables(obs, f'y12_{i}'))
                    ptolemy.append( self.obstruction[obs][i][2] * self.C(f'zpp{i}') * self.change_cross_variables(obs, f'y13_{i}') * self.change_cross_variables(obs, f'y02_{i}') - self.change_cross_variables(obs, f'y01_{i}') * self.change_cross_variables(obs, f'y23_{i}'))
                
                ptolemy.append(non_zero_polynomial - 1)
                
                
                ptolemy.append(c0 - 1)

                ideals.append(self.C.ideal(ptolemy))
                
            return ideals
        

        

In [92]:
from pkg_resources import parse_version
from sage.version import version as s_version
sage_version = parse_version(s_version)
if sage_version < parse_version('9.0'):
    raise ValueError("you need a more recent version of SageMath")
from sage.libs.pari import pari
from cypari2 import Gen
from cypari2.pari_instance import (prec_words_to_dec,
                                   prec_words_to_bits,
                                   prec_bits_to_dec,
                                   prec_dec_to_bits)

from sage.all import PariError
shut_up = lambda: None
speak_up = lambda: None

class Volume():
    def __init__(self, monodromy):
        self.bundle = Punctured_Torus_Bundle(monodromy)
        self.varieties = self.make_varieties()
    
    #Now we map the gluing equations to our reduced dimensional space and generate the variety over which our solutions exist.
    def make_varieties(self):
        varieties = []

        for obs in range(len(self.bundle.obstruction)):
            try:
                varieties.append(self.bundle.cross_ideals[obs].variety(QQbar))
            except:
                print(f'Obstruction class {obs} is not dimension 0')
                varieties.append([])

        return varieties          

    def _dilog(self, z):
        return pari("dilog(%s)" % z)

    def _convert_to_pari_float(self, z):

        if type(z) == Gen and z.type() in ['t_INT', 't_FRAC']:
            return z * pari('1.0')

        return pari(z)

    def _volume(self, z):

        z = self._convert_to_pari_float(float(z.real()) + float(z.imag())*I)

        return (1-z).arg() * z.abs().log() + self._dilog(z).imag()

    def vol(self):
        for obs in range(len(self.bundle.obstruction)):
            print(f'Obstruction class {obs}')
            for var in self.varieties[obs]:
                vol = 0
                for i in range(self.bundle.period):  
                    vol +=  abs(self._volume(var[self.bundle.C(f'z{i}')]))
                #print(self.bundle.C(f'z{i}'), '=', var[self.bundle.C(f'z{i}')])
                print(vol)
                print()

In [74]:
Volume([1,2,1,2]).vol()

Defining x0, x1, x2, x3, x4, x5, const
Defining c0, z0, zp0, zpp0, c1, z1, zp1, zpp1, c2, z2, zp2, zpp2, c3, z3, zp3, zpp3, c4, z4, zp4, zpp4, c5, z5, zp5, zpp5, c
Obstruction class 1 is not dimension 0
Obstruction class 0
Obstruction class 1
Obstruction class 2
0.E-19

0.E-19

0.E-19

0.E-19

Obstruction class 3
-5.42101086242752 E-20

5.42101086242752 E-20

5.42101086242752 E-20

-5.42101086242752 E-20

-5.42101086242752 E-20

5.42101086242752 E-20

5.42101086242752 E-20

-5.42101086242752 E-20

Obstruction class 4
Obstruction class 5
Obstruction class 6
Obstruction class 7


In [93]:
B = Punctured_Torus_Bundle([1,2,1,2])
V = Volume([1,2,1,2])

Defining x0, x1, x2, x3, x4, x5, const
Defining c0, z0, zp0, zpp0, c1, z1, zp1, zpp1, c2, z2, zp2, zpp2, c3, z3, zp3, zpp3, c4, z4, zp4, zpp4, c5, z5, zp5, zpp5, c
Defining x0, x1, x2, x3, x4, x5, const
Defining c0, z0, zp0, zpp0, c1, z1, zp1, zpp1, c2, z2, zp2, zpp2, c3, z3, zp3, zpp3, c4, z4, zp4, zpp4, c5, z5, zp5, zpp5, c
Obstruction class 1 is not dimension 0


In [94]:
for v in V.varieties:
    print('New Obstruction Class:')
    for k in v:
        print('New Representation:')
        for i in range(B.period):    
            print(f'z{i} =', k[B.C(f'z{i}')])
            print(f'zp{i} =', k[B.C(f'zp{i}')])
            print(f'zpp{i} =', k[B.C(f'zpp{i}')])
            print()
        print()
    print()
#             print(k[B.C(f'z{i}')] == 1 - 1/k[B.C(f'zp{i}')])
#             print(k[B.C(f'zp{i}')] == 1 - 1/k[B.C(f'zpp{i}')])
#             print(k[B.C(f'zpp{i}')] == 1 - 1/k[B.C(f'z{i}')])
#             print()
           

New Obstruction Class:

New Obstruction Class:

New Obstruction Class:
New Representation:
z0 = -0.50000000000000000?
zp0 = 0.6666666666666667?
zpp0 = -3

z1 = -0.50000000000000000?
zp1 = 0.6666666666666667?
zpp1 = -3

z2 = -0.3333333333333334?
zp2 = 0.75000000000000000?
zpp2 = 4

z3 = -0.50000000000000000?
zp3 = 0.6666666666666667?
zpp3 = 3

z4 = -0.50000000000000000?
zp4 = 0.6666666666666667?
zpp4 = 3

z5 = -0.3333333333333334?
zp5 = 0.75000000000000000?
zpp5 = 4


New Representation:
z0 = -0.50000000000000000?
zp0 = 0.6666666666666667?
zpp0 = -3

z1 = -0.50000000000000000?
zp1 = 0.6666666666666667?
zpp1 = -3

z2 = -0.3333333333333334?
zp2 = 0.75000000000000000?
zpp2 = 4

z3 = -0.50000000000000000?
zp3 = 0.6666666666666667?
zpp3 = 3

z4 = -0.50000000000000000?
zp4 = 0.6666666666666667?
zpp4 = 3

z5 = -0.3333333333333334?
zp5 = 0.75000000000000000?
zpp5 = 4


New Representation:
z0 = -0.50000000000000000?
zp0 = 0.6666666666666667?
zpp0 = -3

z1 = -0.50000000000000000?
zp1 = 0.66666666

In [95]:
V.vol()

Obstruction class 0
Obstruction class 1
Obstruction class 2
0.E-19

0.E-19

0.E-19

0.E-19

Obstruction class 3
3.38313868803218

3.38313868803218

3.38313868803218

3.38313868803218

3.38313868803218

3.38313868803218

3.38313868803218

3.38313868803218

Obstruction class 4
Obstruction class 5
Obstruction class 6
Obstruction class 7


In [4]:
import snappy

S = snappy.twister.Surface('S_1_1')

M = S.bundle(monodromy='a*B*B*a*B')

N = snappy.Manifold('b++RLLRLL')

# P = snappy.Manifold('4_1')



# T = snappy.Triangulation(M)

# T.num_tetrahedra()
# print()
# print(M.ptolemy_variety(2, 1))
# print()
# print(snappy.OrientableCuspedCensus.identify(M))
# print()
# # print(N.ptolemy_variety(2, 1))

# print(N.ptolemy_variety(2,5))
print(N.volume())
N.ptolemy_variety(2,'all').compute_solutions(engine='sage').cross_ratios().volume_numerical()

# snappy.OrientableCuspedCensus.identify(N).ptolemy_variety(2,'all')






5.33348956689812


[[], [], [], [[0.E-19, 0.E-19], [0.E-19, 0.E-19]], [[-5.33348956689814, 5.33348956689814, 5.33348956689814, -5.33348956689814], [-5.33348956689814, 5.33348956689814, 5.33348956689814, -5.33348956689814], NonZeroDimensionalComponent(dimension = 1)], [], [], [[0.E-19, 0.E-19, 0.E-19, 0.E-19], [0.E-19, 0.E-19, 0.E-19, 0.E-19]]]

In [311]:
Punctured_Torus_Bundle([1,2,1,2]).obstruction[1]

Defining x0, x1, x2, x3, x4, x5, const
Defining c0, z0, zp0, zpp0, c1, z1, zp1, zpp1, c2, z2, zp2, zpp2, c3, z3, zp3, zpp3, c4, z4, zp4, zpp4, c5, z5, zp5, zpp5, c


[[1, -1, 1, 1, 1, 1],
 [-1, 1, 1, 1, 1, 1],
 [1, -1, 1, 1, 1, 1],
 [1, -1, 1, 1, 1, 1],
 [-1, 1, 1, 1, 1, 1],
 [1, -1, 1, 1, 1, 1]]

In [305]:
Punctured_Torus_Bundle([2,1]).period_array

Defining x0, x1, x2, const
Defining c0, z0, zp0, zpp0, c1, z1, zp1, zpp1, c2, z2, zp2, zpp2, c


[0, 0, 1]

In [ ]:
B = Punctured_Torus_Bundle([1,1])
V = B.cross_ideals[1].variety(QQbar)

for i in range(B.period):    
    print(f'z{i} =', V[0][B.C(f'z{i}')])
    print(f'zp{i} =', V[0][B.C(f'zp{i}')])
    print(f'zpp{i} =', V[0][B.C(f'zpp{i}')])

5.07470803204827